# B6: Combine monthly HANPPharv components

In [ ]:
import yaml
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
from rasterio.enums import Resampling
import rioxarray
from geocube.api.core import make_geocube
import matplotlib.pyplot as plt

import sys
sys.path.append(str(Path('../src')))
from grid_utils import create_target_grid

with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)
    
for k, v in cfg['paths'].items():
    cfg["paths"][k] = Path(v)#.expanduser().resolve()

cfg["data_root"] = Path(cfg["data_root"])

In [ ]:
year = 2015
def combine_harvests(year):
    da_HANPPharv_crop_1year = xr.open_dataset(cfg["paths"]["output"] / f'HANPPharv_crops_{year}.nc')\
        .to_array(dim='crop').sum(dim='crop')
    da_HANPPharv_grass_1year = xr.open_dataset(cfg["paths"]["output"] / f'HANPPharv_graze_{year}.nc')["graze"]
    da_HANPPharv_foresty_1year = xr.open_dataset(cfg["paths"]["output"] / f'HANPPharv_forest_{year}.nc')["forest"]
    da_HANPPharv_total = da_HANPPharv_crop_1year + da_HANPPharv_grass_1year + da_HANPPharv_foresty_1year
    da_HANPPharv_total = da_HANPPharv_total\
        .assign_coords(month=pd.date_range(start=f'{year}-01-01', end=f'{year}-12-31', freq='ME'))\
        .rename({"month": "time"})
    del da_HANPPharv_crop_1year, da_HANPPharv_grass_1year, da_HANPPharv_foresty_1year
    return da_HANPPharv_total


In [ ]:
lst_da_harv = []
for year in range(2010, 2025):
    da_harv = combine_harvests(year)
    lst_da_harv.append(da_harv)

In [ ]:
da_harv_merge = xr.concat(lst_da_harv, dim='time')

In [ ]:
da_harv_merge\
    .to_dataset(name='HANPPharv')\
    .to_netcdf(
        cfg["paths"]["output"] / 'monthly/HANPPharv.nc',
        encoding={'HANPPharv': {'zlib': True, 'complevel': 5}}
    )